[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/EelcoHoogendoorn/numga/blob/main/examples/surfaces/projective_loop/projective_loop.ipynb)

# Loop Subdivision of Weighted Points

This example computes Loop subdivision of a triangulated surface whose vertices are weighted projective points. Loop subdivision splits every triangle into four, moves each existing vertex to a weighted average of itself and its neighbours, and inserts a new vertex on each edge between its endpoints and the two opposite corners so that repeated steps converge to a smooth surface. When each vertex is modelled as a point of projective space carrying its own weight and the subdivision averages those weighted points directly, two consequences follow. First, each vertex weight acts as a handle on the surface, pulling the surface toward a heavy vertex and releasing it from a light one. Second, subdivision commutes with perspective, so refining and then viewing in perspective gives the same surface as viewing in perspective and then refining, whereas averaging Euclidean positions does not.

The vertices are points `Point` of projective geometric algebra, and one Loop step `step * controls` is a linear map from the coarser topology's vertex sites to the finer topology's sites that composes across repeated steps into one map. A central collineation `Point + centre * (axis & Point) * strength` maps points to points at each site while the subdivision step acts across sites, so the two commute.

In the homogeneous-coordinate notation of computer-aided geometric design, each weighted vertex reads as $(w\,\mathbf p,\ w)$, a Loop step reads as a subdivision matrix $S$ acting across the vertices from the left, and a projective transformation reads as a $4 \times 4$ homography matrix $H$ acting on each vertex from the right, so rational Loop subdivision satisfies $(S V) H^{\mathsf T} = S (V H^{\mathsf T})$ whereas dividing rows by $w$ before multiplying by $S$ inserts a diagonal weight-rescaling matrix that does not commute with $S$.

In [ ]:
# In Colab, numga and its examples, from GitHub.
import sys

if "google.colab" in sys.modules:
    %pip install -q "numga[examples] @ git+https://github.com/EelcoHoogendoorn/numga"

In [ ]:
%matplotlib inline
import numpy as np

from numga import NumpyContext, stack
from numga.algebras import PGA3D as ga
from numga.topology import Topology
from examples.util.animation import save_figure, show_animation
from examples.surfaces.projective_loop import render

mv = NumpyContext(ga).multivector
Scalar = ga.gatype.scalar()                         # Scalar+
Point = ga.gatype.antivector()                      # Antivector-

## Composing Loop steps across vertex sites

In one Loop step, an existing vertex blends a share of itself with equal shares of its edge neighbours according to its valence, while each new edge vertex takes three eighths of each endpoint and one eighth of each opposite corner across the edge (Loop, 1987). Because these shares depend only on how the triangles are connected, each step is a linear map from the vertex sites of the coarse topology to those of the fine one, listing the existing vertices followed by one new vertex per edge. Composing three steps produces a single map from the octahedron's six vertex sites to the 258 sites of the refined surface.

In subdivision-stencil notation, an existing vertex of valence $n$ retains $1 - n\beta$ of itself and takes $\beta$ of each neighbour with $\beta = \tfrac1n\big(\tfrac58 - (\tfrac38 + \tfrac14\cos\tfrac{2\pi}{n})^2\big)$, while each edge vertex takes $\tfrac38(v_0 + v_1) + \tfrac18(v_2 + v_3)$.

In [ ]:
# The octahedron's six vertices, with `top_vertex` at the +z apex, its eight counter-clockwise faces,
# and the number of Loop subdivision steps.
octahedron = np.array([[1, 0, 0], [-1, 0, 0], [0, 1, 0], [0, -1, 0], [0, 0, 1], [0, 0, -1]])
top_vertex = 4
octahedron_faces = np.array([[0, 2, 4], [2, 1, 4], [1, 3, 4], [3, 0, 4], [2, 0, 5], [1, 2, 5], [3, 1, 5], [0, 3, 5]])
n_loop_steps = 3

In [ ]:
# The octahedron's topology, and three Loop steps of it with no vertices pinned, composed into one map
# from the octahedron's vertex sites to those of the refined topology.
octahedron_topology = Topology(len(octahedron), octahedron_faces)
V, E, F = octahedron_topology.n_elements
refined_topology = octahedron_topology.loop_subdivided(np.zeros(V, dtype=bool), n_loop_steps)
subdivision = refined_topology.subdivision                                      # Scalar+[V_refined <- V]
print((subdivision * Scalar).gatype, subdivision.shape)

## Vertex weights as handles on the surface

Applying the composed map refines the octahedron across uniform weights, a fourfold heavier top vertex, and a fourfold lighter top vertex as a single batch, then sweeps the top weight continuously between those bounds.

In [ ]:
# Vertex weights for the three static cases (uniform, heavy top vertex, light top vertex) and parameters
# for the log-cosine weight sweep animation.
weights = np.array([[1.0, 1.0, 1.0, 1.0, 1.0, 1.0], [1.0, 1.0, 1.0, 1.0, 4.0, 1.0], [1.0, 1.0, 1.0, 1.0, 0.25, 1.0]])
weight_bounds = (0.25, 4.0)
n_sweep_frames = 72
frame_duration_ms = 40

In [ ]:
controls = (mv.point(octahedron) * weights).field()                             # [3] Point-[V]
surfaces = subdivision * controls                                               # [3] Point-[V_refined]

save_figure(render.weights_figure(controls, octahedron_topology, surfaces, refined_topology), "projective_loop_weights");

With equal weights on the left, the subdivided surface matches ordinary Loop subdivision of Euclidean positions inside the dotted control cage. Making the top vertex four times heavier in the middle pulls the surface upward toward the apex, while making it four times lighter on the right lets the surface sink back toward the equator.

In [ ]:
# Sweep the top weight from light to heavy along a cosine in log-weight; the animation plays forward and back.
n_half_frames = n_sweep_frames // 2 + 1
phase = np.linspace(0, np.pi, n_half_frames)                                    # [n_half_frames]
top_weight = np.exp(np.log(weight_bounds).mean() - np.log(weight_bounds[1] / weight_bounds[0]) / 2 * np.cos(phase))  # [n_half_frames]
sweep_weights = np.where(np.arange(V) == top_vertex, top_weight[:, None], 1.0)  # [n_half_frames, V]
sweep_controls = ((mv.point(octahedron) * weight).field() for weight in sweep_weights)

show_animation(render.weights_animation(((control, subdivision * control) for control in sweep_controls), octahedron_topology, refined_topology),
               "projective_loop_sweep", frame_duration_ms, forward_then_back=True)

## Commuting with a perspective collineation

A central collineation shifts each point along its line through a centre by `Point + centre * (axis & Point) * strength`, leaving the axis plane fixed while changing the weight of every point off it. Because the Loop map acts across vertex sites while the collineation acts on the point at each site, refining before or after the collineation produces the exact same surface. Normalising points to unit weight before subdividing discards the weight change induced by the collineation, so the two orders diverge.

In [ ]:
# Stretch and offset placing the octahedron in front of a central collineation with centre at the origin
# and axis plane z = 0.
stretch = np.array([1.0, 1.0, 1.5])
offset = np.array([0.0, 0.0, 1.6])
centre = mv.w.dual()                                                            # Antivector-
axis = mv.z                                                                     # Vector+
strength = 1.0

In [ ]:
rest_control = mv.point(octahedron * stretch + offset).field()                  # [] Point-[V]
# The central collineation shifts each point along its line through the centre in proportion to its signed
# distance from the axis plane.
collineation = Point + centre * (axis & Point) * strength                       # [] Point- <- Point-
perspective_control = collineation(rest_control)                                # [] Point-[V]
weighted_orders = stack([subdivision * perspective_control, collineation(subdivision * rest_control)])  # [2] Point-[V_refined]
rest_positions = rest_control.normalized()                                      # [] Point-[V]
perspective_positions = perspective_control.normalized()                        # [] Point-[V]
positional_orders = stack([subdivision * perspective_positions, collineation(subdivision * rest_positions)])  # [2] Point-[V_refined]
# Normalising each refined surface to unit weight makes the norm of the join between the two orders equal
# to their Euclidean separation.
weighted_unit = weighted_orders.normalized()                                    # [2] Point-[V_refined]
positional_unit = positional_orders.normalized()                                # [2] Point-[V_refined]
weighted_gaps = (weighted_unit[0] & weighted_unit[1]).norm()                    # [] Scalar+[V_refined]
positional_gaps = (positional_unit[0] & positional_unit[1]).norm()              # [] Scalar+[V_refined]

print(f"largest gap between the orders of Euclidean subdivision: {positional_gaps.to_array().max():.3f}")
save_figure(render.orders_figure(perspective_control, octahedron_topology, weighted_orders, positional_orders, refined_topology),
            "projective_loop_perspective");

Both panels share one colour scale measuring the Euclidean distance to the surface obtained in the opposite order. On the left, subdividing weighted points stays uniformly blue because the two orders coincide everywhere; on the right, subdividing normalised Euclidean positions turns bright orange where the collineation changes the vertex weights most strongly.

In [ ]:
# checks
assert (weighted_gaps < 1e-12).all()
assert (positional_gaps > 1e-2).any()
# With all weights equal, weighted projective subdivision agrees with Euclidean position subdivision.
assert ((surfaces[0] & surfaces[0].normalized()).norm() < 1e-12).all()